# 05.2 — Extract content from documents

Three ways to read a document, side by side on the same file:

| | Grounding | Output | Cost shape |
|---|---|---|---|
| **Document Intelligence** | boxes + spans + per-field confidence | rigid typed JSON, and markdown from layout | per page |
| **Content Understanding** | spans on extracted fields | **your** field schema, plus markdown | per content unit + model tokens |
| **Multimodal LLM** | none | whatever you ask for | per token |

**Cost.** Per-call, not hourly — nothing here bills while you sleep. But
`prebuilt-layout` is roughly ten times `prebuilt-read` per page, so do not loop it
over a large corpus casually. This lab processes about six pages.

The final cell deletes every analyzer and blob it creates.

**Prerequisites:** a Foundry / AI Services resource with `Cognitive Services User`
granted to you, a chat and an embedding deployment, and (for section 5) Content
Understanding available in your region with default model deployments bound.

## 1. Configuration

Document Intelligence and Content Understanding both live on the **Foundry (AI
Services) resource** and share its endpoint and RBAC. If `.env` does not name them
separately we fall back to the Foundry endpoint, which is usually correct.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import (
    cfg,
    credential,
    token_provider,
    chat_client,
    blob_service_client,
    ask,
)

OUT = pathlib.Path.cwd() / "lab_output"
OUT.mkdir(exist_ok=True)
(OUT / ".gitignore").write_text("*\n", encoding="utf-8")

PREFIX = "ai103-52"
CONTAINER = f"{PREFIX}-src"

FOUNDRY = cfg.require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DI_ENDPOINT = (cfg.get("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT") or FOUNDRY).rstrip("/")
CU_ENDPOINT = (cfg.get("AZURE_CONTENT_UNDERSTANDING_ENDPOINT") or FOUNDRY).rstrip("/")
CU_API_VERSION = "2025-11-01"  # GA

print("document intelligence :", DI_ENDPOINT)
print("content understanding :", CU_ENDPOINT)
print("output                :", OUT)

## 2. Generate the source documents

Two files, chosen to expose the difference between the models:

- **`invoice.pdf`** — headings, a table, a total. `prebuilt-read` will flatten the
  table; `prebuilt-layout` will recover it; `prebuilt-invoice` will name the fields.
- **`chart.png`** — a bar chart with axis labels. There is no text layer at all, and
  the *meaning* is in the geometry, not the words.

In [ ]:
def build_pdf(path: pathlib.Path, lines: list[str]) -> None:
    """Minimal hand-built one-page PDF with a real text layer (no dependencies)."""
    parts = ["BT", "/F1 11 Tf", "50 750 Td", "15 TL"]
    for line in lines:
        safe = line.replace("\\", r"\\").replace("(", r"\(").replace(")", r"\)")
        parts.append(f"({safe}) Tj T*")
    parts.append("ET")
    stream = "\n".join(parts).encode("latin-1")

    objects = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
        b"/Resources << /Font << /F1 5 0 R >> >> /Contents 4 0 R >>",
        b"<< /Length " + str(len(stream)).encode() + b" >>\nstream\n" + stream + b"\nendstream",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
    ]

    buf = bytearray(b"%PDF-1.4\n")
    offsets = []
    for i, obj in enumerate(objects, start=1):
        offsets.append(len(buf))
        buf += f"{i} 0 obj\n".encode() + obj + b"\nendobj\n"
    xref_at = len(buf)
    buf += f"xref\n0 {len(objects) + 1}\n".encode() + b"0000000000 65535 f \n"
    for off in offsets:
        buf += f"{off:010d} 00000 n \n".encode()
    buf += (
        f"trailer\n<< /Size {len(objects) + 1} /Root 1 0 R >>\nstartxref\n{xref_at}\n".encode()
        + b"%%EOF\n"
    )
    path.write_bytes(bytes(buf))


INVOICE_LINES = [
    "CONTOSO LTD.",
    "440 Terry Avenue North, Redmond, WA 98052",
    "",
    "INVOICE",
    "",
    "Invoice Number: INV-2026-0417",
    "Invoice Date: 2026-03-14",
    "Due Date: 2026-04-13",
    "Purchase Order: PO-88213",
    "",
    "Bill To: Fabrikam Industrial Services",
    "1200 Harbour Road, Tacoma, WA 98402",
    "",
    "Line Items",
    "",
    "Description                      Qty     Unit Price     Amount",
    "CX-4400 Industrial Controller      3        1250.00    3750.00",
    "CX-4400 Firmware Support (yr)      3         180.00     540.00",
    "Onsite Commissioning               1         900.00     900.00",
    "",
    "Subtotal                                              5190.00",
    "Tax (8.8 percent)                                      456.72",
    "Total Due                                             5646.72",
    "",
    "Payment Terms",
    "",
    "Net 30. Late payments accrue interest at 1.5 percent per month.",
    "Remit to account 4471-99820 quoting the invoice number.",
    "",
    "Warranty Note",
    "",
    "Controllers carry a 36 month limited warranty from commissioning.",
    "Warranty is void if the enclosure seal is broken.",
]

build_pdf(OUT / "invoice.pdf", INVOICE_LINES)
print("wrote invoice.pdf:", (OUT / "invoice.pdf").stat().st_size, "bytes")

In [ ]:
from PIL import Image, ImageDraw

img = Image.new("RGB", (720, 420), "white")
d = ImageDraw.Draw(img)
d.line([(80, 350), (680, 350)], fill="black", width=2)   # x axis
d.line([(80, 40), (80, 350)], fill="black", width=2)     # y axis

series = [("Q1", 120), ("Q2", 185), ("Q3", 240), ("Q4", 205)]
for i, (label, value) in enumerate(series):
    x = 130 + i * 130
    d.rectangle([x, 350 - value, x + 70, 350], fill="#4a6fa5")
    d.text((x + 22, 358), label, fill="black")
    d.text((x + 18, 350 - value - 16), str(value), fill="black")

d.text((250, 12), "RMA volume by quarter, 2026", fill="black")
d.text((92, 30), "units", fill="black")
img.save(OUT / "chart.png")
print("wrote chart.png")
img

## 3. The Document Intelligence model ladder

Same file, three models, increasing price and increasing structure.

`azure-ai-documentintelligence` 1.0.0 (v4.0 API) is the current package. The older
`azure-ai-formrecognizer` uses `DocumentAnalysisClient` and has no markdown output.

Auth is Entra ID — `DefaultAzureCredential` with *Cognitive Services User* on the
Foundry resource.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult

try:  # renamed between the 1.0.0 betas and GA
    from azure.ai.documentintelligence.models import DocumentContentFormat as _Fmt
except ImportError:  # pragma: no cover
    from azure.ai.documentintelligence.models import ContentFormat as _Fmt

di = DocumentIntelligenceClient(endpoint=DI_ENDPOINT, credential=credential())
PDF_BYTES = (OUT / "invoice.pdf").read_bytes()


def analyze(model_id: str, data: bytes, **kwargs) -> AnalyzeResult:
    """Run a Document Intelligence model over raw bytes."""
    poller = di.begin_analyze_document(
        model_id, AnalyzeDocumentRequest(bytes_source=data), **kwargs
    )
    return poller.result()


read = analyze("prebuilt-read", PDF_BYTES)
print("prebuilt-read")
print("-" * 70)
print("pages      :", len(read.pages or []))
print("languages  :", [l.locale for l in (read.languages or [])][:3])
print("handwritten:", bool(read.styles and any(s.is_handwritten for s in read.styles)))
print("tables     :", len(read.tables or []), " <- read does not detect tables")
print("\nfirst 400 characters of flat content:")
print((read.content or "")[:400])

In [ ]:
layout = analyze("prebuilt-layout", PDF_BYTES)

print("prebuilt-layout")
print("-" * 70)
print("tables         :", len(layout.tables or []))
print("selection marks:", sum(len(p.selection_marks or []) for p in (layout.pages or [])))
print("paragraphs     :", len(layout.paragraphs or []))

roles = {}
for p in layout.paragraphs or []:
    roles.setdefault(p.role or "(body)", []).append(p.content)
print("\nparagraph roles detected:")
for role, items in roles.items():
    print(f"  {role:<16} {len(items):>3}   e.g. {items[0][:50]!r}")

print("\nEvery value is grounded. One paragraph, fully traced:")
p = (layout.paragraphs or [])[0]
print("  content        :", p.content[:60])
print("  spans          :", [(s.offset, s.length) for s in (p.spans or [])])
for br in p.bounding_regions or []:
    print("  page", br.page_number, "polygon:", [round(v, 2) for v in br.polygon])

`spans` index into the flat `content` string; `boundingRegions` are inches on a
page. Together they are what lets a reviewer click a value and see it highlighted
on the original. **A multimodal model gives you neither**, which is the whole
argument for this service in a regulated workflow.

In [ ]:
invoice = analyze("prebuilt-invoice", PDF_BYTES)

print("prebuilt-invoice — typed, named fields with per-field confidence")
print("-" * 70)

THRESHOLD = 0.80  # auto-accept above this; queue for human review below

for doc in invoice.documents or []:
    print("doc type:", doc.doc_type, " overall confidence:", doc.confidence)
    for name in [
        "InvoiceId", "InvoiceDate", "DueDate", "VendorName",
        "CustomerName", "PurchaseOrder", "SubTotal", "TotalTax", "InvoiceTotal",
    ]:
        field = (doc.fields or {}).get(name)
        if field is None:
            print(f"  {name:<16} (not found)")
            continue
        value = (
            field.get("valueString")
            or field.get("valueDate")
            or field.get("valueCurrency")
            or field.get("valueNumber")
            or field.get("content")
        )
        conf = field.get("confidence")
        verdict = "auto" if (conf or 0) >= THRESHOLD else "REVIEW"
        print(f"  {name:<16} {str(value)[:38]:<38} conf={conf}  -> {verdict}")

    items = (doc.fields or {}).get("Items")
    if items and items.get("valueArray"):
        print(f"\n  Items[] — {len(items['valueArray'])} rows recovered as structured data")
        for row in items["valueArray"][:4]:
            props = row.get("valueObject") or {}
            desc = (props.get("Description") or {}).get("valueString", "")
            amt = (props.get("Amount") or {}).get("valueCurrency", {})
            print(f"    {desc[:44]:<44} {amt.get('amount')}")

> **This is the operational pattern the exam asks about.** Confidence is
> **per field**, so the review threshold is per field too — `InvoiceTotal` deserves
> a stricter bar than `VendorName`. Tune the thresholds against a labelled sample;
> do not pick them by feel. And note the failure mode confidence does **not** catch:
> a perfectly-read number from the wrong column scores 0.99.

Custom models slot into exactly the same call. Train an extraction model in
Document Intelligence Studio on 5+ labelled samples and then:

```python
analyze("<your-custom-model-id>", PDF_BYTES)   # identical call shape
```

For mixed batches, run a **custom classification** model first to decide the type,
then route to the matching extraction model. **Template** build mode learns
positions (5 samples, fast, brittle to layout change); **neural** learns structure
(more samples, generalises across layouts).

## 4. Layout → markdown → RAG chunks

This is the highest-leverage thing in the unit. `output_content_format="markdown"`
turns layout analysis into a document with real heading structure and real tables,
which you can then chunk **on headings** instead of on character counts.

In [ ]:
md_result = analyze("prebuilt-layout", PDF_BYTES, output_content_format=_Fmt.MARKDOWN)
markdown = md_result.content or ""
(OUT / "invoice.md").write_text(markdown, encoding="utf-8")

print("markdown output, first 1200 characters")
print("=" * 70)
print(markdown[:1200])
print("=" * 70)
print("\nheading lines recovered:")
for line in markdown.splitlines():
    if line.startswith("#"):
        print("  ", line)
print("\npipe-table lines:", sum(1 for l in markdown.splitlines() if l.strip().startswith("|")))

In [ ]:
import re


def chunk_markdown(md: str, source_uri: str, max_chars: int = 1500) -> list[dict]:
    """Structural chunking: split on headings, keep the heading path on every chunk.

    The heading path is not decoration. A chunk that begins 'Net 30. Late payments
    accrue interest...' is neither retrievable nor citable. 'INVOICE > Payment
    Terms > Net 30...' is both.
    """
    chunks, path, buf = [], [], []

    def flush():
        body = "\n".join(buf).strip()
        if not body:
            return
        heading_path = " > ".join(path) if path else "(document root)"
        # split oversized sections on blank lines rather than mid-sentence
        pieces, current = [], ""
        for para in body.split("\n\n"):
            if len(current) + len(para) > max_chars and current:
                pieces.append(current)
                current = para
            else:
                current = f"{current}\n\n{para}" if current else para
        if current:
            pieces.append(current)
        for piece in pieces:
            chunks.append(
                {
                    "heading_path": heading_path,
                    "source_uri": source_uri,
                    "text": f"{heading_path}\n\n{piece.strip()}",
                }
            )

    for line in md.splitlines():
        m = re.match(r"^(#{1,6})\s+(.*)$", line)
        if m:
            flush()
            buf = []
            level = len(m.group(1))
            path = path[: level - 1] + [m.group(2).strip()]
        else:
            buf.append(line)
    flush()
    return chunks


chunks = chunk_markdown(markdown, source_uri="lab_output/invoice.pdf")
print(f"{len(chunks)} structural chunks\n")
for c in chunks:
    print(f"[{c['heading_path']}]  {len(c['text'])} chars")
    print("   ", " ".join(c["text"].split())[:110], "...\n")

Compare that with what fixed-size chunking would have produced from
`prebuilt-read`'s flat text: chunks that begin mid-table, headings orphaned from
their bodies, and no path to prepend. These chunks drop straight into the 05.1
index with `heading_path` and `source_uri` as retrievable fields — which is the
*grounded* half of "clean, grounded representations".

> **Exam note.** Symptom "our RAG answers are wrong on PDFs with tables" →
> layout-aware extraction to markdown, then heading-based chunking. Not a bigger
> chunk size, not a bigger embedding model.

Inside Azure AI Search the same capability is the
`Microsoft.Skills.Util.DocumentIntelligenceLayoutSkill`, with `markdownHeaderDepth`
choosing how deep the section split goes.

## 5. Content Understanding analyzers

One API, four modalities, and **you** define the output shape.

The lab calls the REST API directly with a bearer token: nothing extra to install,
Entra ID auth, and the wire format — which is what the exam describes — stays
visible. A preview `azure-ai-contentunderstanding` package also exists.

Both `PUT` (create analyzer) and `POST :analyze` are **long-running operations**:
you get 202 plus an `Operation-Location` header and poll it.

In [ ]:
import json, time, requests

get_token = token_provider("https://cognitiveservices.azure.com/.default")


def cu_headers(content_type: str | None = None) -> dict:
    h = {"Authorization": f"Bearer {get_token()}"}
    if content_type:
        h["Content-Type"] = content_type
    return h


def cu_poll(operation_url: str, timeout_s: int = 300) -> dict:
    """Poll an Operation-Location until it stops being NotStarted/Running."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        r = requests.get(operation_url, headers=cu_headers(), timeout=60)
        r.raise_for_status()
        body = r.json()
        status = str(body.get("status", "")).lower()
        if status not in ("notstarted", "running"):
            return body
        time.sleep(2)
    raise TimeoutError(f"operation did not complete: {operation_url}")


# Smoke test: can we reach the service at all?
probe = requests.get(
    f"{CU_ENDPOINT}/contentunderstanding/analyzers?api-version={CU_API_VERSION}",
    headers=cu_headers(),
    timeout=60,
)
CU_AVAILABLE = probe.status_code == 200
print("Content Understanding reachable:", CU_AVAILABLE, f"(HTTP {probe.status_code})")
if not CU_AVAILABLE:
    print(probe.text[:400])
    print("\n404 -> not available in this region, or the endpoint is not a Foundry")
    print("       AI Services resource. 401/403 -> you need Cognitive Services User.")
else:
    names = [a.get("analyzerId") for a in probe.json().get("value", [])][:10]
    print("existing analyzers:", names or "(none)")

### 5a. Make the file reachable

`:analyze` takes `{"inputs": [{"url": ...}]}` and the service fetches the URL — so
the file has to be reachable from Azure, not from your laptop. We upload to blob
storage and mint a short-lived **user-delegation SAS**.

A user-delegation SAS is signed with a key derived from *your Entra token*, not
from the storage account key. That keeps the lab keyless: there is no account key
anywhere, and the SAS expires in an hour whatever you do.

In [ ]:
import datetime as dt
from azure.storage.blob import generate_blob_sas, BlobSasPermissions

bsc = blob_service_client()
try:
    bsc.create_container(CONTAINER)
except Exception as exc:
    print("container:", type(exc).__name__)

container = bsc.get_container_client(CONTAINER)
start = dt.datetime.now(dt.timezone.utc) - dt.timedelta(minutes=5)
expiry = start + dt.timedelta(hours=1)
udk = bsc.get_user_delegation_key(start, expiry)  # derived from your Entra token


def upload_and_sign(path: pathlib.Path) -> str:
    with path.open("rb") as fh:
        container.upload_blob(name=path.name, data=fh, overwrite=True)
    sas = generate_blob_sas(
        account_name=cfg["AZURE_STORAGE_ACCOUNT"],
        container_name=CONTAINER,
        blob_name=path.name,
        user_delegation_key=udk,
        permission=BlobSasPermissions(read=True),
        start=start,
        expiry=expiry,
    )
    return f"{container.url}/{path.name}?{sas}"


INVOICE_URL = upload_and_sign(OUT / "invoice.pdf")
CHART_URL = upload_and_sign(OUT / "chart.png")
print("invoice url:", INVOICE_URL.split("?")[0], "+ 1h SAS")
print("chart   url:", CHART_URL.split("?")[0], "+ 1h SAS")

### 5b. Define an analyzer schema

All three generation methods in one schema, deliberately:

| Field | `method` | Why this method |
|---|---|---|
| `VendorName`, `InvoiceNumber`, `TotalDue` | **`extract`** | The value is literally on the page. Grounded, auditable. |
| `DocumentCategory`, `PaymentRisk` | **`classify`** | One of an enumerated set. The `enum` constrains the model. |
| `ActionSummary` | **`generate`** | Not present in the document at all — the model writes it. |
| `LineItems` | **`extract`** + `array`/`object` | Repeating structures need a nested schema. |

`description` is a **prompt**, not documentation. It is the main lever you have on
quality. Vague descriptions produce vague fields.

In [ ]:
DOC_ANALYZER_ID = f"{PREFIX}-invoice-analyzer"

doc_analyzer = {
    "description": "AI-103 lab: invoice triage with extract, classify, and generate",
    "baseAnalyzerId": "prebuilt-documentAnalyzer",
    "config": {"returnDetails": True},
    "fieldSchema": {
        "fields": {
            "VendorName": {
                "type": "string",
                "method": "extract",
                "description": "Legal name of the company issuing the invoice.",
            },
            "InvoiceNumber": {
                "type": "string",
                "method": "extract",
                "description": "The invoice identifier, e.g. INV-2026-0417.",
            },
            "TotalDue": {
                "type": "number",
                "method": "extract",
                "description": "Final amount payable including tax, as a number.",
            },
            "DocumentCategory": {
                "type": "string",
                "method": "classify",
                "description": "What kind of document this is.",
                "enum": ["invoice", "receipt", "purchase order", "statement", "other"],
            },
            "PaymentRisk": {
                "type": "string",
                "method": "classify",
                "description": (
                    "Risk that this invoice is paid late, judged from the payment "
                    "terms and any penalty clauses."
                ),
                "enum": ["low", "medium", "high"],
            },
            "ActionSummary": {
                "type": "string",
                "method": "generate",
                "description": (
                    "Two sentences an accounts-payable clerk could act on: what is "
                    "owed, by when, and anything unusual about the terms."
                ),
            },
            "LineItems": {
                "type": "array",
                "method": "extract",
                "description": "One entry per billed line.",
                "items": {
                    "type": "object",
                    "properties": {
                        "Description": {"type": "string", "method": "extract"},
                        "Quantity": {"type": "number", "method": "extract"},
                        "Amount": {"type": "number", "method": "extract"},
                    },
                },
            },
        }
    },
}

print(json.dumps(doc_analyzer, indent=2)[:900], "...")

In [ ]:
created_analyzers = []


def cu_create_analyzer(analyzer_id: str, body: dict) -> dict | None:
    url = f"{CU_ENDPOINT}/contentunderstanding/analyzers/{analyzer_id}?api-version={CU_API_VERSION}"
    r = requests.put(url, headers=cu_headers("application/json"), json=body, timeout=60)
    if r.status_code not in (200, 201, 202):
        print(f"create {analyzer_id} failed: HTTP {r.status_code}")
        print(r.text[:500])
        return None
    created_analyzers.append(analyzer_id)
    op = r.headers.get("Operation-Location")
    result = cu_poll(op) if op else r.json()
    print(f"analyzer {analyzer_id}: {result.get('status', 'created')}")
    if str(result.get("status", "")).lower() == "failed":
        print(json.dumps(result, indent=2)[:600])
    return result


def cu_analyze(analyzer_id: str, url: str) -> dict | None:
    endpoint = (
        f"{CU_ENDPOINT}/contentunderstanding/analyzers/{analyzer_id}"
        f":analyze?api-version={CU_API_VERSION}"
    )
    r = requests.post(
        endpoint,
        headers=cu_headers("application/json"),
        json={"inputs": [{"url": url}]},
        timeout=60,
    )
    if r.status_code not in (200, 202):
        print(f"analyze {analyzer_id} failed: HTTP {r.status_code}")
        print(r.text[:500])
        return None
    op = r.headers.get("Operation-Location")
    return cu_poll(op) if op else r.json()


if CU_AVAILABLE:
    cu_create_analyzer(DOC_ANALYZER_ID, doc_analyzer)
else:
    print("skipped - Content Understanding not reachable")

In [ ]:
def show_fields(result: dict) -> None:
    if not result:
        return
    print("status:", result.get("status"))
    for content in (result.get("result") or {}).get("contents", []):
        md = content.get("markdown") or ""
        print("\nmarkdown (first 300 chars):")
        print(" ", " ".join(md.split())[:300])
        print("\nfields:")
        for name, field in (content.get("fields") or {}).items():
            value = next(
                (v for k, v in field.items() if k.startswith("value")),
                field.get("content"),
            )
            conf = field.get("confidence")
            spans = field.get("spans")
            grounded = "grounded" if spans else "ungrounded"
            rendered = json.dumps(value)[:120] if isinstance(value, (list, dict)) else str(value)[:120]
            print(f"  {name:<18} {rendered}")
            print(f"  {'':<18} conf={conf}  {grounded}")


if CU_AVAILABLE:
    show_fields(cu_analyze(DOC_ANALYZER_ID, INVOICE_URL))
else:
    print("skipped")

Look at which fields carry `spans` and which do not. `extract` fields point back
into the content; `generate` fields cannot, because the value is not in the
document. **That is the audit boundary.** If a downstream process has to justify a
value to an auditor, that value must come from `extract`.

### 5c. The same shape, a different modality

The analyzer contract does not change across modalities — only the base analyzer
and the fields you care about. This one reads the chart image, where the meaning is
in the *geometry*, not in the text.

In [ ]:
IMAGE_ANALYZER_ID = f"{PREFIX}-chart-analyzer"

image_analyzer = {
    "description": "AI-103 lab: read a business chart",
    "baseAnalyzerId": "prebuilt-imageAnalyzer",
    "config": {"returnDetails": True},
    "fieldSchema": {
        "fields": {
            "Title": {
                "type": "string",
                "method": "extract",
                "description": "The chart title as printed.",
            },
            "ChartType": {
                "type": "string",
                "method": "classify",
                "enum": ["bar", "line", "pie", "scatter", "table", "other"],
            },
            "PeakPeriod": {
                "type": "string",
                "method": "generate",
                "description": "Which labelled period has the highest value, and its value.",
            },
            "Narrative": {
                "type": "string",
                "method": "generate",
                "description": (
                    "One sentence describing the trend, suitable for indexing as "
                    "searchable text alongside the image."
                ),
            },
        }
    },
}

if CU_AVAILABLE:
    cu_create_analyzer(IMAGE_ANALYZER_ID, image_analyzer)
    show_fields(cu_analyze(IMAGE_ANALYZER_ID, CHART_URL))
else:
    print("skipped — schema above is the exam material either way")

`Narrative` is the piece that closes the loop back to unit 05.1. Images cannot be
embedded usefully as pixels by a text embedding model — but a generated narrative
*can* be, so you index the narrative and keep the image as the citation. The same
trick handles audio and video: run

| Modality | Base analyzer | You index |
|---|---|---|
| Audio | `prebuilt-audioAnalyzer` / `prebuilt-callCenter` | transcript segments + speaker + timestamps |
| Video | `prebuilt-videoAnalyzer` / `prebuilt-videoSearch` | chapter segments + transcript + keyframe descriptions + `start`/`end` |

…and index the extraction, not the media. That is the correct architecture for the
"ingest audio and video" bullet, and the reason Azure AI Search alone cannot do it.

**Pro mode** (preview) extends this to reasoning across multiple input files and
reference data in one analysis, at higher latency and cost.

## 6. The third option: just ask the model

For contrast, the same extraction with a multimodal chat model and a JSON schema.
It is fast, flexible, and needs no analyzer — and it gives you nothing to audit.

In [ ]:
# We feed the model the layout markdown rather than the raw PDF, which is itself the
# lesson: even the "just use the model" path works far better on extracted markdown.
llm_json = chat_client().chat.completions.create(
    model=cfg["MODEL_MINI"],
    temperature=0,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": (
                "Extract invoice data. Reply with JSON only, keys: VendorName, "
                "InvoiceNumber, TotalDue, PaymentRisk, ActionSummary."
            ),
        },
        {"role": "user", "content": markdown[:6000]},
    ],
)
print(llm_json.choices[0].message.content)
print("\nusage:", llm_json.usage.total_tokens, "tokens")

print(
    """
Notice what is missing from that response:
  - no confidence on any field
  - no page number, no bounding box
  - no span back into the source text
  - no guarantee the same input produces the same output next month

For an exploratory question that is fine. For an accounts-payable workflow that a
regulator can audit, it is disqualifying - which is the entire reason Document
Intelligence still exists in a world with capable multimodal models.
"""
)

## 7. The decision table

Commit this to memory; it is the most likely single question from this unit.

| Requirement in the scenario | Choose |
|---|---|
| Bounding boxes, per-field confidence, review queue, audit | **Document Intelligence** |
| A well-known document type (invoice, receipt, ID, tax form) | **Document Intelligence** prebuilt |
| A form Microsoft has never seen, high volume, stable layout | **Document Intelligence** custom (template) |
| Same form, many layout variants | **Document Intelligence** custom (neural), or classify-then-extract |
| Markdown with headings and tables to feed RAG | **Document Intelligence** `prebuilt-layout`, or the layout skill in Search |
| Documents **and** images **and** audio **and** video through one API | **Content Understanding** |
| A schema that is easier to *describe* than to label | **Content Understanding** |
| Output shaped for an agent to consume directly | **Content Understanding** |
| Summaries, inferences, categories that are not literally on the page | **Content Understanding** `generate` / `classify` |
| One-off exploration, no structure or provenance needed | **Multimodal LLM** |
| Open-ended visual reasoning with no fixed output shape | **Multimodal LLM** |

And the composite answer the exam likes best: **Document Intelligence layout →
markdown → heading chunks → Azure AI Search index with integrated vectorization →
agent with a search tool.** Every stage grounded, every stage traceable.

---

## Exercise

Solutions are in [quiz.md](quiz.md).

1. **Cost the ladder.** Run `prebuilt-read` and `prebuilt-layout` over the same PDF,
   time both, and count what each returns (`pages`, `tables`, `paragraphs`,
   `styles`). Then state, in one sentence per case, when paying ~10x for layout is
   *not* worth it.

2. **Break `extract`.** Add a field `ProjectManagerName` with `method: "extract"` to
   the document analyzer and re-run it. The invoice does not contain one. Record
   what comes back. Then change the method to `generate` and re-run. Explain why
   the second result is more dangerous than the first.

3. **Confidence-driven routing.** Write `route(doc, thresholds)` that takes the
   `prebuilt-invoice` result and a per-field threshold dict, and returns
   `("auto", fields)` or `("review", [list of fields below threshold])`. Use a
   stricter bar for `InvoiceTotal` than for `VendorName` and justify the gap.

4. **Close the loop with 05.1.** Take the `chunks` list from section 4 and push it
   into a small Azure AI Search index with `heading_path` and `source_uri` as
   retrievable fields. Query `"what happens if I pay late"` and show that the
   returned chunk carries a citable heading path. Delete the index afterwards.

In [ ]:
# Your work here

---

## Cleanup

Analyzers persist on the resource until deleted, and each one occupies a slot
against the per-resource analyzer limit. Nothing here bills hourly, but leaving
half-built analyzers around is how you hit that limit later and cannot work out
why.

In [ ]:
for analyzer_id in created_analyzers:
    url = f"{CU_ENDPOINT}/contentunderstanding/analyzers/{analyzer_id}?api-version={CU_API_VERSION}"
    try:
        r = requests.delete(url, headers=cu_headers(), timeout=60)
        print(f"deleted analyzer {analyzer_id}: HTTP {r.status_code}")
    except Exception as exc:
        print(f"skipped analyzer {analyzer_id}: {type(exc).__name__}")

try:
    blob_service_client().delete_container(CONTAINER)
    print("deleted container", CONTAINER)
except Exception as exc:
    print("skipped container:", type(exc).__name__)

print("\nNothing in this unit bills hourly - Document Intelligence and Content")
print("Understanding are per-call. If you ran unit 05.1, the Azure AI Search")
print("SERVICE is still billing. See 99_teardown.")

## Check yourself

[quiz.md](quiz.md) — 15 questions on the model ladder, markdown chunking, analyzer
schemas, generation methods, and the three-way service choice.

## Next

[90 — Full study-guide summary and mock exam](../../90_summary/README.md)